# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/houchia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/houchia/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/houchia/Desktop/AIE8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/houchia/Desktop/AIE8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/houchia/Desktop/AIE8/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [14]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 150, relationships: 712)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [15]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/59 [00:00<?, ?it/s]

Property 'headlines' already exists in node '5e5427'. Skipping!
Property 'headlines' already exists in node '33f2d1'. Skipping!
Property 'headlines' already exists in node 'a39d97'. Skipping!
Property 'headlines' already exists in node '67267d'. Skipping!
Property 'headlines' already exists in node '5c9e90'. Skipping!
Property 'headlines' already exists in node '425967'. Skipping!
Property 'headlines' already exists in node '69d5ef'. Skipping!
Property 'headlines' already exists in node 'a4b04a'. Skipping!
Property 'headlines' already exists in node 'd46911'. Skipping!
Property 'headlines' already exists in node 'e13f5d'. Skipping!
Property 'headlines' already exists in node '6c3737'. Skipping!
Property 'headlines' already exists in node 'a32491'. Skipping!
Property 'headlines' already exists in node 'e63618'. Skipping!
Property 'headlines' already exists in node '22994f'. Skipping!
Property 'headlines' already exists in node '31c3ae'. Skipping!
Property 'headlines' already exists in n

Applying HeadlineSplitter:   0%|          | 0/150 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/110 [00:00<?, ?it/s]

...

Applying CustomNodeFilter:   0%|          | 0/21 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/140 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '67267d'. Skipping!
Property 'summary_embedding' already exists in node '22994f'. Skipping!
Property 'summary_embedding' already exists in node '6c3737'. Skipping!
Property 'summary_embedding' already exists in node 'e63618'. Skipping!
Property 'summary_embedding' already exists in node '33f2d1'. Skipping!
Property 'summary_embedding' already exists in node '5c9e90'. Skipping!
Property 'summary_embedding' already exists in node 'd46911'. Skipping!
Property 'summary_embedding' already exists in node 'a4b04a'. Skipping!
Property 'summary_embedding' already exists in node '69d5ef'. Skipping!
Property 'summary_embedding' already exists in node 'a32491'. Skipping!
Property 'summary_embedding' already exists in node 'a39d97'. Skipping!
Property 'summary_embedding' already exists in node 'e13f5d'. Skipping!
Property 'summary_embedding' already exists in node 'cfe84f'. Skipping!
Property 'summary_embedding' already exists in node '5e5427'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 211, relationships: 6746)

We can save and load our knowledge graphs as follows.

In [16]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 211, relationships: 6746)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [18]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5), # one chunk
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25), # synthesize from multiple chunks
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25), # synthesize from multiple chunks
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

**Answer:**

- **SingleHopSpecificQuerySynthesizer**: Creates questions that can be answered from one single chunk of data
- **MultiHopAbstractQuerySynthesizer**: Creates questions that require combining information from multiple chunks and making abstract connections. Asks about patterns, relationships, and conceptual connections.
- **MultiHopSpecificQuerySynthesizer**: Creates questions that require combining specific facts (e.g., exact numbers, dates, or facts) from multiple chunks. Answer "what," "when," "how many" with concrete details.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [19]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"Handa et al., 2025?",[Introduction ChatGPT launched in November 202...,"The paper by Handa et al., 2025, reports stati...",single_hop_specifc_query_synthesizer
1,US what is?,[Table 1: ChatGPT daily message counts (millio...,The context explains that the US is mentioned ...,single_hop_specifc_query_synthesizer
2,How is ChatGPT utilized across different occup...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,How does the term 'Writing' relate to the usag...,[Conclusion This paper studies the rapid growt...,"In the context provided, 'Writing' is identifi...",single_hop_specifc_query_synthesizer
4,how many 18 billion messages sent,[Introduction ChatGPT launched in November 202...,"By July 2025, 18 billion messages were being s...",single_hop_specifc_query_synthesizer
5,How do user behaviors and message classificati...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context indicates that user behaviors and ...,multi_hop_abstract_query_synthesizer
6,How do the classifications of chatbot messages...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context explains that messages sent to Cha...,multi_hop_abstract_query_synthesizer
7,How has the rapid growth of ChatGPT in low- an...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth of ChatGPT, especially in low...",multi_hop_abstract_query_synthesizer
8,What is the signifcance of July 2025 in the co...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"July 2025 is significant because, by this date...",multi_hop_specific_query_synthesizer
9,section how chatgpt use vary by occupation and...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context shows that chatgpt use varies by o...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [20]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '9bf7e6'. Skipping!
Property 'summary' already exists in node 'bf93f5'. Skipping!
Property 'summary' already exists in node '717247'. Skipping!
Property 'summary' already exists in node '99157a'. Skipping!
Property 'summary' already exists in node '504e8c'. Skipping!
Property 'summary' already exists in node '58381b'. Skipping!
Property 'summary' already exists in node '1f291e'. Skipping!
Property 'summary' already exists in node 'a1c8ec'. Skipping!
Property 'summary' already exists in node 'c730f9'. Skipping!
Property 'summary' already exists in node 'ab7f00'. Skipping!
Property 'summary' already exists in node 'b5aec2'. Skipping!
Property 'summary' already exists in node 'a0dd63'. Skipping!
Property 'summary' already exists in node '1f4d5e'. Skipping!
Property 'summary' already exists in node '5102b4'. Skipping!
Property 'summary' already exists in node '0af7a0'. Skipping!
Property 'summary' already exists in node '2ebcee'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'c730f9'. Skipping!
Property 'summary_embedding' already exists in node 'a1c8ec'. Skipping!
Property 'summary_embedding' already exists in node '1f291e'. Skipping!
Property 'summary_embedding' already exists in node 'b5aec2'. Skipping!
Property 'summary_embedding' already exists in node '99157a'. Skipping!
Property 'summary_embedding' already exists in node '9bf7e6'. Skipping!
Property 'summary_embedding' already exists in node 'a0dd63'. Skipping!
Property 'summary_embedding' already exists in node '504e8c'. Skipping!
Property 'summary_embedding' already exists in node '58381b'. Skipping!
Property 'summary_embedding' already exists in node '717247'. Skipping!
Property 'summary_embedding' already exists in node 'bf93f5'. Skipping!
Property 'summary_embedding' already exists in node 'ab7f00'. Skipping!
Property 'summary_embedding' already exists in node '1f4d5e'. Skipping!
Property 'summary_embedding' already exists in node '0af7a0'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [21]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the usage of 2025 impact the adoption...,[Introduction ChatGPT launched in November 202...,The context discusses the rapid growth and ado...,single_hop_specifc_query_synthesizer
1,Tell me about Claude and how it is used in cha...,[Table 1: ChatGPT daily message counts (millio...,The context provides information about ChatGPT...,single_hop_specifc_query_synthesizer
2,Could you explain the significance of SOC2 cod...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,How does the term 'Doing' relate to user inter...,[Conclusion This paper studies the rapid growt...,"In the context of ChatGPT usage, 'Doing' refer...",single_hop_specifc_query_synthesizer
4,how AI effects society and user message classi...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context explains that ChatGPT launched in ...,multi_hop_abstract_query_synthesizer
5,How do LLMs like ChatGPT show AI capabiltys an...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, based on a Large Language Model (LLM)...",multi_hop_abstract_query_synthesizer
6,How does the rapid global diffusion of AI tech...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_abstract_query_synthesizer
7,How do the changing usage patterns from June 2...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that from June 2024 to June 202...,multi_hop_abstract_query_synthesizer
8,"In June 2025, how does the usage of ChatGPT di...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By June 2025, ChatGPT's usage shows that non-w...",multi_hop_specific_query_synthesizer
9,"hOw do OpenAI's ChatGPT usage patterns, especi...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context indicates that ChatGPT, developed ...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [22]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [23]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [24]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [25]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [26]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [27]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [28]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [29]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [31]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [32]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, including generative AI like ChatGPT, in various ways both at work and outside of work. AI is employed to perform workplace tasks by either augmenting or automating human labor. Users seek AI to produce writing, software code, spreadsheets, and other digital products. Additionally, AI serves roles both as co-workers that produce output and as co-pilots that provide advice and improve human problem-solving productivity. Key user intents with AI include asking for information and advice, doing tasks (such as creating digital content), and expressing themselves. Moreover, generative AI is highly flexible and distinguishes itself from traditional search engines by its ability to create diverse digital outputs. There is also evidence of AI use in areas like therapy/companionship and games/role play, though these are smaller portions of use cases.\n\nIn summary, people are using AI to:\n- Augment or automate work tasks\n- Produce digital c

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [33]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [34]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates the correctness of the answer by comparing the generated response against a reference answer. It uses LangSmith's built-in QA evaluator to determine if the answer is factually correct and relevant to the question.

- `labeled_helpfulness_evaluator`: Evaluates how helpful the response is to the user by taking into account the correct reference answer. It specifically measures whether the submission provides useful information that addresses the user's question effectively.

- `dopeness_evaluator`: Evaluates the style and quality of the response to determine if it's "dope, lit, cool" rather than generic. It measures the creativity, engagement, and overall appeal of the response beyond just factual correctness.

## LangSmith Evaluation

In [35]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'artistic-rain-81' at:
https://smith.langchain.com/o/59e7f291-de60-42a7-a056-0fe9209518e1/datasets/bf3a92a5-4aae-4b9e-86d2-272592a4afef/compare?selectedSessions=1fe38e3f-4a6e-44b2-bc94-fc88791421e7




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how openai chatgpt used by people in openai an...,"Based on the provided context, OpenAI's ChatGP...",None,the context shows that chatgpt launched in nov...,0,0,0,5.343712,680fa229-6d78-4c54-83d5-3c9b60d2380d,9ace819a-fbb3-4661-beab-1ea98821b2e1
1,Considering the rapid growth of ChatGPT usage ...,The context indicates that between June 2024 a...,None,"The data from June 2024, as presented in <2-ho...",1,1,0,7.083508,e879adda-b9ce-4293-af84-587bace7e01f,0342d5bf-44a1-41fd-8cad-b166262976e6
2,"hOw do OpenAI's ChatGPT usage patterns, especi...","Based on the provided context, OpenAI's ChatGP...",None,"The context indicates that ChatGPT, developed ...",1,1,0,4.419745,a2436ff1-5d5b-4b59-a84a-b571e30d71c1,c8a20352-3c1a-4f2b-b790-bd268bd98115
3,"In June 2025, how does the usage of ChatGPT di...","In June 2025, non-work-related messages accoun...",None,"By June 2025, ChatGPT's usage shows that non-w...",1,1,0,4.211374,454cb60b-7bd3-4b3b-a44e-2bd9ba645e99,15498bcb-e01c-4c83-93a6-56ec7578166a
4,How do the changing usage patterns from June 2...,"Based on the provided context, the usage patte...",None,The data shows that from June 2024 to June 202...,1,1,0,7.378317,bae64782-5440-4cf4-8960-713b327d1517,ec5d59ad-ae5e-4970-b90b-35ae7d7fd6d8
5,How does the rapid global diffusion of AI tech...,The context indicates that ChatGPT experienced...,None,"By July 2025, ChatGPT had been used weekly by ...",1,1,0,6.126644,572aa70a-25a7-4e69-a690-23da737f12f2,7cc68a34-3b16-4335-9a79-bd45e16b8594
6,How do LLMs like ChatGPT show AI capabiltys an...,Based on the context provided:\n\nLLMs like Ch...,None,"ChatGPT, based on a Large Language Model (LLM)...",1,1,0,6.058358,74c20d66-9b7b-4774-a110-285ec2a7fa01,caed917f-b6ea-439f-9712-ca2e5814c3df
7,how AI effects society and user message classi...,"Based on the provided context, AI, specificall...",None,The context explains that ChatGPT launched in ...,1,1,0,6.941771,151e7570-0eb0-44ab-a400-82d3db30ed11,1fffb7b0-d3c7-49e7-9447-cd35161e7e80
8,How does the term 'Doing' relate to user inter...,"According to the provided context, ""Doing"" mes...",None,"In the context of ChatGPT usage, 'Doing' refer...",1,1,0,3.924018,2694207f-f839-4525-87b3-6d297260ce70,29423db9-0922-46f4-a165-257ea1499e10
9,Could you explain the significance of SOC2 cod...,"Based on the provided context, SOC2 code 11 co...",None,Variation by Occupation Figure 23 presents var...,1,1,0,4.905554,45ce3d88-3fc3-49ef-b29f-a2c8bf32fbc3,741338a1-31c4-4781-82b9-53445a1072f6


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [36]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [37]:
rag_documents = docs

In [38]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

**Answer:** Modifying chunk size affects RAG performance in several key ways:

1. **Context Completeness**: Larger chunks (1000 vs 500) provide more complete context for each retrieved document. This means the LLM has access to more comprehensive information to answer questions, potentially reducing the need for multi-hop reasoning across multiple chunks.

2. **Retrieval Precision**: Smaller chunks allow for more precise retrieval - if a question is about a specific detail, a smaller chunk containing just that detail is more likely to be retrieved. Larger chunks might include irrelevant information that dilutes the signal.

3. **Information Density**: Larger chunks can contain more related information in one place, which is beneficial for complex questions that require understanding relationships between concepts within the same document section.

4. **Computational Efficiency**: Larger chunks mean fewer total chunks to process, but each chunk contains more text for the LLM to process. This creates a trade-off between retrieval efficiency and processing overhead.

5. **Overlap Considerations**: With larger chunks, the 50-character overlap becomes proportionally smaller, potentially missing important context that spans chunk boundaries.

In this case, increasing from 500 to 1000 characters likely improves performance because the documents contain complex, interconnected information that benefits from having more complete context in each retrieved chunk.

In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

**Answer:** Modifying the embedding model affects RAG performance in several critical ways:

1. **Embedding Quality**: Different embedding models have varying capabilities in capturing semantic meaning. The upgrade from `text-embedding-3-small` to `text-embedding-3-large` provides higher-quality vector representations that better capture the nuanced relationships between concepts in the text.

2. **Retrieval Accuracy**: Better embeddings lead to more accurate similarity matching between queries and document chunks. This means the retriever is more likely to find the most relevant information for a given question, reducing irrelevant or partially relevant chunks.

3. **Semantic Understanding**: Larger embedding models typically have better understanding of context, synonyms, and conceptual relationships. This helps in retrieving chunks that are semantically similar even if they don't share exact keywords with the query.

4. **Dimensionality**: `text-embedding-3-large` produces higher-dimensional embeddings (typically 3072 dimensions vs 1536 for the small model), which can capture more fine-grained semantic distinctions and improve retrieval precision.

5. **Domain Adaptation**: Different embedding models may perform better on specific types of content. The larger model might be better at understanding the specific domain and terminology used in the AI use cases documents.

6. **Query-Document Matching**: Better embeddings improve the cosine similarity calculations between user queries and document chunks, leading to more relevant retrieved context for the LLM to work with.

In this case, upgrading to `text-embedding-3-large` likely improves performance because it provides more sophisticated semantic understanding, leading to better retrieval of relevant context chunks for answering questions about AI use cases.

In [40]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [41]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [42]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [43]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

"Alright, buckle up because here comes the AI hustle breakdown straight from the data vault!\n\nPeople ain’t just pushing buttons—they’re leveraging generative AI, like ChatGPT, as a top-tier *advisor* and *research assistant* in the money-making game. According to Collis and Brynjolfsson (2025), AI isn’t just a tool cranking out tasks; it’s a secret sauce boosting worker output by sharpening decision-making where it counts most—knowledge-intensive gigs. Think of it as a high-octane brain booster, fueling smarter, faster, and better choices that translate into cold hard cash.\n\nThis AI sidekick elevates workers from mere doers to strategic power players, leading to mind-blowing productivity spikes. The estimates say US users treasure this so much they'd shell out nearly $98 a month just to keep the AI in their corner, which stacks up to a jaw-dropping $97 billion yearly surplus in value. Simply put: AI isn’t just cutting costs; it’s rewriting the rulebook on how knowledge work dollars

Finally, we can evaluate the new chain on the same test set!

In [44]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'impressionable-trousers-21' at:
https://smith.langchain.com/o/59e7f291-de60-42a7-a056-0fe9209518e1/datasets/bf3a92a5-4aae-4b9e-86d2-272592a4afef/compare?selectedSessions=66f58629-cdfe-4576-8fee-bce0245c1776




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how openai chatgpt used by people in openai an...,"Alright, buckle up for a turbo-charged dive in...",None,the context shows that chatgpt launched in nov...,0,0,1,7.011668,680fa229-6d78-4c54-83d5-3c9b60d2380d,7caff949-0593-42ae-b5d7-82d09cf07f32
1,Considering the rapid growth of ChatGPT usage ...,"Oh, buckle up — this growth story isn’t just n...",None,"The data from June 2024, as presented in <2-ho...",1,1,1,7.167197,e879adda-b9ce-4293-af84-587bace7e01f,b8a5a05b-a1c8-4040-83e6-641729ee11b6
2,"hOw do OpenAI's ChatGPT usage patterns, especi...","Oh snap, let’s drop some knowledge bombs on ho...",None,"The context indicates that ChatGPT, developed ...",1,1,1,6.408076,a2436ff1-5d5b-4b59-a84a-b571e30d71c1,03ae0928-04a3-44a2-84b4-1cdb65497578
3,"In June 2025, how does the usage of ChatGPT di...","Yo, check this out — by June 2025, non-work Ch...",None,"By June 2025, ChatGPT's usage shows that non-w...",1,1,1,4.695995,454cb60b-7bd3-4b3b-a44e-2bd9ba645e99,8a176811-4da5-4568-9a20-1904b6416f8a
4,How do the changing usage patterns from June 2...,"Yo, buckle up—this dataset is spilling straigh...",None,The data shows that from June 2024 to June 202...,1,1,1,5.892328,bae64782-5440-4cf4-8960-713b327d1517,270270fc-a2dc-48e6-80aa-dae50dd9ff07
5,How does the rapid global diffusion of AI tech...,"Yo, the ChatGPT saga is straight fire when it ...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,6.845334,572aa70a-25a7-4e69-a690-23da737f12f2,e06e714f-d0fd-4c2b-a509-3a841d8a8611
6,How do LLMs like ChatGPT show AI capabiltys an...,"Yo, buckle up, ‘cause here’s the lowdown on ho...",None,"ChatGPT, based on a Large Language Model (LLM)...",1,1,1,6.740362,74c20d66-9b7b-4774-a110-285ec2a7fa01,52d4d3bb-3317-4ce1-ad5c-2bb8cef3dfba
7,how AI effects society and user message classi...,"Alright, buckle up — here’s the lowdown on how...",None,The context explains that ChatGPT launched in ...,1,1,1,13.940118,151e7570-0eb0-44ab-a400-82d3db30ed11,cc396c1b-606a-4c24-ac36-c0f6cbd80086
8,How does the term 'Doing' relate to user inter...,"Yo, buckle up because the term **'Doing'** is ...",None,"In the context of ChatGPT usage, 'Doing' refer...",1,1,1,4.812426,2694207f-f839-4525-87b3-6d297260ce70,731db499-45c6-43af-988f-b34c2d65bf3d
9,Could you explain the significance of SOC2 cod...,"Alright, let’s drop some knowledge bombs about...",None,Variation by Occupation Figure 23 presents var...,1,1,1,7.137089,45ce3d88-3fc3-49ef-b29f-a2c8bf32fbc3,8e1e0a21-8683-4c1d-a386-4ed1ea8af8e9


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

**Answer:**

![Comparison of RAG Chain Evaluations](./comparison.png)

**Analysis of Performance Changes:**

1. **QA Evaluator (Correctness)**: Both chains show very similar scores of approximately **0.82**, indicating that factual accuracy remained consistent. This suggests that the changes (larger chunks, better embeddings, dopeness prompt) didn't compromise the factual correctness of the responses.

2. **Labeled Helpfulness Evaluator (Helpfulness)**: Both chains also show very similar scores of approximately **0.82**, indicating that helpfulness remained consistent. The improvements in context retrieval and response style didn't negatively impact the overall helpfulness of the responses.

3. **Dopeness Evaluator (Dopeness)**: This shows the most dramatic improvement. The default chain scored very low at approximately **0.02**, while the dopeness chain achieved a perfect score of approximately **1.0**. This massive improvement demonstrates that the dopeness prompt successfully transformed generic responses into engaging, "dope" content as intended.

**Overall Performance Trends:**

The modifications had a clear trade-off pattern:

**Positive Changes:**
- **Dopeness**: Massive improvement (0.02 → 1.0) due to the explicit dopeness prompt encouraging creative, engaging responses
- **Maintained Quality**: Both correctness and helpfulness remained stable, showing that the improvements didn't come at the cost of factual accuracy or usefulness

**Performance Costs:**
- **Latency Increased**: P50 latency increased from ~5.0s to ~6.8s, and P99 latency increased from ~7.5s to ~12.8s
- This latency increase is likely due to the larger chunks requiring more processing time and the more complex dopeness prompt requiring additional LLM computation

**Conclusion**: The changes successfully achieved the primary goal of making responses more engaging and "dope" while maintaining factual accuracy and helpfulness, but at the cost of increased response time.
